# TransformG4

## **Módulo**: Inteligencia de Negocios

**Integrantes:**
- Cristian Gonzalez
- Oscar Amagua
- Rafael Espinosa

## Grupo
Nro. 4

### Descripción del proyecto

Este proyecto analiza datos de COVID-19 en México con el fin de construir un modelo dimensional en esquema estrella (Star Schema). El presente notebook documenta la etapa **Transform** del proceso ETL: limpieza, normalización, generación de identificadores, integración entre fuentes y validación de resultados.

Las fuentes utilizadas son:
- **PostgreSQL:** catálogos dimensionales (`cat_*`).
- **JSON:** `5 Sector.json` y `8 Tipo_Paciente.json`.
- **CSV:** `covid-19_general_MX.csv` como tabla de hechos principal.

## Importación de Dependencias

In [1]:
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
from sqlalchemy import create_engine

# Fase de Extracción de Datos

En esta fase se obtienen los datos desde las fuentes definidas para el proyecto:

- **Base de datos PostgreSQL:** catálogos de entidad, sexo, nacionalidad, origen, resultado y respuesta binaria.
- **Archivos JSON:** catálogos de sector y tipo de paciente.
- **Archivo CSV:** registros generales de casos COVID-19 en México.

### Manejo de Variables de Entorno

In [2]:
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")


### Seguridad y variables de entorno

Las credenciales y parámetros de conexión se cargan desde el archivo `.env` mediante `python-dotenv`. Esta práctica evita exponer usuarios y contraseñas en el código fuente y permite reutilizar el mismo notebook en distintos entornos sin modificar la lógica de transformación.

## Cargando nuestra Base de Datos

In [3]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

df_entidades = pd.read_sql("select * from cat_entidad_federativa", engine)
df_nacionalidad = pd.read_sql("select * from cat_nacionalidad", engine)
df_origen = pd.read_sql("select * from cat_tipo_usmer", engine)
df_resultado = pd.read_sql("select * from cat_resultado_laboratorio", engine)
df_sexo = pd.read_sql("select * from cat_sexo", engine)
df_respuesta = pd.read_sql("select * from cat_respuesta_binaria", engine)

## Carga de Datos Tipo Json

In [4]:
df_sector = pd.read_json("data/5 Sector.json")
df_tipopaciente = pd.read_json("data/8 Tipo_Paciente.json")

## Carga de Datos tipo CSV

In [5]:
df_covidgeneral = pd.read_csv("data/covid-19_general_MX.csv")

## Análisis de Cada DataFrame

El objetivo de este análisis es identificar la estructura, calidad y relaciones entre las fuentes de datos. Esta evaluación orienta las reglas de limpieza, la selección de columnas y la integración posterior con el modelo analítico.

### Base de Datos Postgres

In [6]:
catalogos = {
    "entidades": df_entidades,
    "nacionalidad": df_nacionalidad,
    "origen": df_origen,
    "resultado": df_resultado,
    "sexo": df_sexo,
    "respuesta": df_respuesta,
}

resumen_catalogos = []
for nombre, df in catalogos.items():
    resumen_catalogos.append({
        "catalogo": nombre,
        "registros": len(df),
        "columnas": df.shape[1],
        "nulos": int(df.isnull().sum().sum()),
        "duplicados": int(df.duplicated().sum()),
    })

pd.DataFrame(resumen_catalogos)

,catalogo,registros,columnas,nulos,duplicados
0,entidades,36,3,0,0
1,nacionalidad,3,2,0,0
2,origen,3,2,0,0
3,resultado,3,2,0,0
4,sexo,3,2,0,0
5,respuesta,5,2,0,0


Los catálogos en PostgreSQL son tablas pequeñas con claves numéricas estables. No presentan nulos ni duplicados relevantes y funcionan como dimensiones para enriquecer la tabla de hechos mediante `merge`.

### Datos Json

In [7]:
resumen_json = []
for nombre, df in {"sector": df_sector, "tipo_paciente": df_tipopaciente}.items():
    resumen_json.append({
        "archivo": nombre,
        "registros": len(df),
        "columnas": ", ".join(df.columns),
        "nulos": int(df.isnull().sum().sum()),
        "duplicados": int(df.duplicated().sum()),
    })

pd.DataFrame(resumen_json)

,archivo,registros,columnas,nulos,duplicados
0,sector,14,"clave, descripcion",0,0
1,tipo_paciente,3,"clave, descripcion",0,0


Los archivos JSON contienen las columnas `clave` y `descripcion`. Corresponden a los catálogos de sector institucional y tipo de paciente utilizados por el CSV principal.

### Archivo de Datos CSV

El archivo `covid-19_general_MX.csv` contiene la tabla de hechos del proyecto. En esta etapa se revisa la estructura y la calidad del dato **en su estado original**, antes de aplicar las reglas de transformación.

En este dataset, los valores faltantes no siempre aparecen como nulos: la Secretaría de Salud utiliza códigos oficiales como `97`, `98`, `99` y la fecha `9999-99-99` para indicar "no aplica", "se ignora" o "no especificado".


In [8]:
print(f"Registros: {len(df_covidgeneral):,}")
print(f"Columnas: {df_covidgeneral.shape[1]}")
print(f"Memoria aproximada: {df_covidgeneral.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

df_covidgeneral.head()

Registros: 879,608
Columnas: 26
Memoria aproximada: 302.8 MB


,Unnamed: 0,SECTOR,ENTIDAD_UM,SEXO,ENTIDAD_RES,TIPO_PACIENTE,FECHA_INGRESO,FECHA_SINTOMAS,FECHA_DEF,INTUBADO,...,INMUSUPR,HIPERTENSION,OTRA_CON,CARDIOVASCULAR,OBESIDAD,RENAL_CRONICA,TABAQUISMO,OTRO_CASO,RESULTADO,UCI
0,0,4,9,2,9,1,2020-03-23,2020-03-22,9999-99-99,97,...,2,2,2,2,2,2,2,99,1,97
1,1,4,9,2,9,2,2020-04-13,2020-04-04,9999-99-99,2,...,2,2,2,2,1,2,2,99,1,2
2,2,4,8,1,8,2,2020-04-15,2020-04-10,2020-04-19,2,...,2,1,2,2,1,2,2,99,1,2
3,3,4,30,1,30,1,2020-04-27,2020-04-17,9999-99-99,97,...,2,2,1,2,2,2,2,99,1,97
4,4,3,15,2,15,2,2020-06-06,2020-06-01,9999-99-99,2,...,2,2,2,2,2,2,2,1,1,2


In [9]:
codigos_especiales = [97, 98, 99]
columnas_binarias = [
    "INTUBADO", "NEUMONIA", "DIABETES", "EPOC", "ASMA", "INMUSUPR",
    "HIPERTENSION", "OTRA_CON", "CARDIOVASCULAR", "OBESIDAD",
    "RENAL_CRONICA", "TABAQUISMO", "UCI",
]

resumen_csv = pd.DataFrame([
    {"indicador": "Total de registros", "valor": f"{len(df_covidgeneral):,}"},
    {"indicador": "Total de columnas", "valor": df_covidgeneral.shape[1]},
    {"indicador": "Registros duplicados", "valor": int(df_covidgeneral.duplicated().sum())},
    {
        "indicador": "FECHA_DEF con codigo 9999-99-99",
        "valor": int((df_covidgeneral["FECHA_DEF"] == "9999-99-99").sum()),
    },
    {
        "indicador": "Interpretacion FECHA_DEF 9999-99-99",
        "valor": "Sin fecha de defuncion registrada (codigo oficial del dataset)",
    },
    {
        "indicador": "Columna Unnamed: 0",
        "valor": "Presente" if "Unnamed: 0" in df_covidgeneral.columns else "No presente",
    },
    {
        "indicador": "Edad promedio",
        "valor": round(df_covidgeneral["EDAD"].mean(), 2),
    },
    {
        "indicador": "Registros con codigos 97/98/99 en variables binarias",
        "valor": int(df_covidgeneral[columnas_binarias].isin(codigos_especiales).any(axis=1).sum()),
    },
])

resumen_csv


,indicador,valor
0,Total de registros,"879,608"
1,Total de columnas,26
2,Registros duplicados,0
3,FECHA_DEF con codigo 9999-99-99,825207
4,Interpretacion FECHA_DEF 9999-99-99,Sin fecha de defuncion registrada (codigo ofic...
5,Columna Unnamed: 0,Presente
6,Edad promedio,42.55
7,Registros con codigos 97/98/99 en variables bi...,708280


**Hallazgos del CSV:**
- La fuente contiene aproximadamente 879 mil registros y 26 columnas, lo que representa un volumen considerable de información para el proceso ETL.
- No se observan valores nulos explícitos, ya que el conjunto de datos utiliza códigos especiales para representar determinadas condiciones o estados, en lugar de dejar los campos vacíos.
- En este conjunto de datos no fue necesario eliminar registros ni imputar valores, debido a que los códigos utilizados forman parte de la definición oficial de la fuente y contienen información relevante para el análisis. Eliminar o reemplazar estos valores podría alterar el significado de los datos y afectar los resultados.
- El valor **FECHA_DEF = 9999-99-99** indica que el paciente no registra una fecha de defunción, por lo que corresponde a un dato válido y no a un error de captura.
- Las variables clínicas están representadas mediante códigos numéricos, por lo que en la etapa de transformación se estandarizarán utilizando los catálogos correspondientes para facilitar su interpretación.
- La columna **Unnamed: 0** será eliminada porque corresponde a un índice generado durante la exportación del archivo y no aporta información útil para el análisis.


### Relaciones entre fuentes

| Campo en hechos | Fuente destino | Campo de unión |
|---|---|---|
| `SECTOR` | `df_sector` | `clave` |
| `TIPO_PACIENTE` | `df_tipopaciente` | `clave` |
| `ENTIDAD_RES` | `df_entidades` | `clave_entidad` |
| `SEXO` | `df_sexo` | `clave` |
| `NACIONALIDAD` | `df_nacionalidad` | `clave` |
| `RESULTADO` | `df_resultado` | `clave` |
| Variables binarias | `df_respuesta` | `clave` |

# Inicio del Proceso de Transformación de los Datos

En esta etapa se aplican reglas de negocio y criterios analíticos para preparar la información del modelo dimensional. El proceso incluye funciones de limpieza, transformación de tipos, generación de identificadores e integración entre tablas de hechos y dimensiones.

## Funciones de Limpieza y Transformación

In [10]:
MAPEO_BINARIO = {
    1: "SI",
    2: "NO",
    97: "NO APLICA",
    98: "SE IGNORA",
    99: "NO ESPECIFICADO",
}

def limpiar_fecha(valor):
    """Convierte codigos oficiales de fecha no aplicable a valor nulo."""
    if pd.isna(valor):
        return None
    texto = str(valor).strip()
    if texto in {"", "9999-99-99", "NaT"}:
        return None
    return texto

def normalizar_codigo_binario(valor):
    """Traduce codigos oficiales 1/2/97/98/99 a descripcion legible."""
    if pd.isna(valor):
        return "NO ESPECIFICADO"
    try:
        clave = int(valor)
    except (TypeError, ValueError):
        return "NO ESPECIFICADO"
    return MAPEO_BINARIO.get(clave, "NO ESPECIFICADO")

def validar_edad(edad):
    """Valida edad y genera grupo etario para analisis."""
    if pd.isna(edad):
        return False, "SIN EDAD"
    try:
        edad_num = int(edad)
    except (TypeError, ValueError):
        return False, "SIN EDAD"
    if edad_num < 0 or edad_num > 120:
        return False, "SIN EDAD"
    if edad_num <= 17:
        grupo = "0-17"
    elif edad_num <= 39:
        grupo = "18-39"
    elif edad_num <= 59:
        grupo = "40-59"
    else:
        grupo = "60+"
    return True, grupo

def agregar_id_secuencial(df, nombre_columna):
    """Genera identificador numerico secuencial para integracion entre tablas."""
    df = df.copy()
    df[nombre_columna] = df.index + 1
    return df


## Transformación de Catálogos PostgreSQL

In [11]:
df_entidades = agregar_id_secuencial(df_entidades, "id_entidad")
df_nacionalidad = agregar_id_secuencial(df_nacionalidad, "id_nacionalidad")
df_origen = agregar_id_secuencial(df_origen, "id_origen")
df_resultado = agregar_id_secuencial(df_resultado, "id_resultado")
df_sexo = agregar_id_secuencial(df_sexo, "id_sexo")
df_respuesta = agregar_id_secuencial(df_respuesta, "id_respuesta")

df_entidades.head()

,clave_entidad,entidad_federativa,abreviatura,id_entidad
0,1,AGUASCALIENTES,AS,1
1,2,BAJA CALIFORNIA,BC,2
2,3,BAJA CALIFORNIA SUR,BS,3
3,4,CAMPECHE,CC,4
4,5,COAHUILA DE ZARAGOZA,CL,5


Se generan identificadores secuenciales (`id_*`) para facilitar la integración con la tabla de hechos. Las claves originales se conservan para realizar los `merge` con el archivo CSV.

## Transformación de Archivos JSON

In [12]:
df_sector = df_sector.drop_duplicates().copy()
df_sector = df_sector[df_sector["clave"].notnull()]
df_sector["clave"] = df_sector["clave"].astype("int64")
df_sector = agregar_id_secuencial(df_sector, "id_sector")

df_tipopaciente = df_tipopaciente.drop_duplicates().copy()
df_tipopaciente = df_tipopaciente[df_tipopaciente["clave"].notnull()]
df_tipopaciente["clave"] = df_tipopaciente["clave"].astype("int64")
df_tipopaciente = agregar_id_secuencial(df_tipopaciente, "id_tipo_paciente")

df_sector

,clave,descripcion,id_sector
0,1,CRUZ ROJA,1
1,2,DIF,2
2,3,ESTATAL,3
3,4,IMSS,4
4,5,IMSS-BIENESTAR,5
5,6,ISSSTE,6
6,7,MUNICIPAL,7
7,8,PEMEX,8
8,9,PRIVADA,9
9,10,SEDENA,10


## Transformación del DataFrame de Hechos

### Selección de columnas relevantes

Se conservan las variables necesarias para el análisis epidemiológico y las uniones con catálogos: demografía, fechas, comorbilidades, resultado de laboratorio y atención hospitalaria. Se elimina `Unnamed: 0` por tratarse de un índice exportado sin valor analítico.

In [13]:
df_covid = df_covidgeneral.copy()

if "Unnamed: 0" in df_covid.columns:
    df_covid = df_covid.drop(columns=["Unnamed: 0"])

columnas_hechos = [
    "SECTOR",
    "ENTIDAD_UM",
    "SEXO",
    "ENTIDAD_RES",
    "TIPO_PACIENTE",
    "FECHA_INGRESO",
    "FECHA_SINTOMAS",
    "FECHA_DEF",
    "INTUBADO",
    "NEUMONIA",
    "EDAD",
    "NACIONALIDAD",
    "DIABETES",
    "EPOC",
    "ASMA",
    "INMUSUPR",
    "HIPERTENSION",
    "OTRA_CON",
    "CARDIOVASCULAR",
    "OBESIDAD",
    "RENAL_CRONICA",
    "TABAQUISMO",
    "OTRO_CASO",
    "RESULTADO",
    "UCI",
]

df_covid = df_covid[columnas_hechos]
df_covid = df_covid[df_covid["ENTIDAD_RES"].notnull()]
df_covid = df_covid.drop_duplicates()

pd.DataFrame([
    {"etapa": "Seleccion de columnas y filtros", "registros": len(df_covid), "columnas": df_covid.shape[1]},
])

,etapa,registros,columnas
0,Seleccion de columnas y filtros,849638,25


### Limpieza de fechas, edad y códigos

In [14]:
for col in ["FECHA_INGRESO", "FECHA_SINTOMAS", "FECHA_DEF"]:
    df_covid[col] = df_covid[col].apply(lambda x: limpiar_fecha(x))

for col in ["FECHA_INGRESO", "FECHA_SINTOMAS", "FECHA_DEF"]:
    df_covid[col] = pd.to_datetime(df_covid[col], errors="coerce")

df_covid[["edad_valida", "grupo_edad"]] = df_covid[["EDAD"]].apply(
    lambda x: validar_edad(x["EDAD"]), axis=1, result_type="expand"
)

columnas_binarias = [
    "INTUBADO",
    "NEUMONIA",
    "DIABETES",
    "EPOC",
    "ASMA",
    "INMUSUPR",
    "HIPERTENSION",
    "OTRA_CON",
    "CARDIOVASCULAR",
    "OBESIDAD",
    "RENAL_CRONICA",
    "TABAQUISMO",
    "UCI",
]

for col in columnas_binarias:
    df_covid[f"{col}_DESC"] = df_covid[col].apply(
        lambda x: normalizar_codigo_binario(x)
    )

df_covid["fallecido"] = df_covid["FECHA_DEF"].notnull()
df_covid["dias_sintomas_ingreso"] = (
    df_covid["FECHA_INGRESO"] - df_covid["FECHA_SINTOMAS"]
).dt.days

df_covid = agregar_id_secuencial(df_covid, "id_caso")
df_covid.head()

,SECTOR,ENTIDAD_UM,SEXO,ENTIDAD_RES,TIPO_PACIENTE,FECHA_INGRESO,FECHA_SINTOMAS,FECHA_DEF,INTUBADO,NEUMONIA,...,HIPERTENSION_DESC,OTRA_CON_DESC,CARDIOVASCULAR_DESC,OBESIDAD_DESC,RENAL_CRONICA_DESC,TABAQUISMO_DESC,UCI_DESC,fallecido,dias_sintomas_ingreso,id_caso
0,4,9,2,9,1,2020-03-23,2020-03-22,NaT,97,2,...,NO,NO,NO,NO,NO,NO,NO APLICA,False,1,1
1,4,9,2,9,2,2020-04-13,2020-04-04,NaT,2,1,...,NO,NO,NO,SI,NO,NO,NO,False,9,2
2,4,8,1,8,2,2020-04-15,2020-04-10,2020-04-19,2,1,...,SI,NO,NO,SI,NO,NO,NO,True,5,3
3,4,30,1,30,1,2020-04-27,2020-04-17,NaT,97,2,...,NO,SI,NO,NO,NO,NO,NO APLICA,False,10,4
4,3,15,2,15,2,2020-06-06,2020-06-01,NaT,2,1,...,NO,NO,NO,NO,NO,NO,NO,False,5,5


Durante la transformación se normalizan fechas con códigos oficiales, se valida la edad y se agrupa en rangos etarios. Los códigos binarios se convierten a descripciones legibles mediante funciones y `apply`. Además, se genera el identificador `id_caso` para cada registro.


## Integración de Hechos con Dimensiones

La tabla de hechos se integra con los catálogos mediante `merge` de tipo `left`, preservando todos los registros del CSV y agregando las claves e identificadores dimensionales correspondientes.

In [15]:
df_covid_clean = df_covid.merge(
    df_sector[["clave", "descripcion", "id_sector"]],
    how="left",
    left_on="SECTOR",
    right_on="clave",
    suffixes=("", "_sector"),
)

df_covid_clean = df_covid_clean.merge(
    df_tipopaciente[["clave", "descripcion", "id_tipo_paciente"]],
    how="left",
    left_on="TIPO_PACIENTE",
    right_on="clave",
    suffixes=("", "_tipo"),
)

df_covid_clean = df_covid_clean.merge(
    df_entidades[["clave_entidad", "entidad_federativa", "id_entidad"]],
    how="left",
    left_on="ENTIDAD_RES",
    right_on="clave_entidad",
)

df_covid_clean = df_covid_clean.merge(
    df_sexo[["clave", "descripcion", "id_sexo"]],
    how="left",
    left_on="SEXO",
    right_on="clave",
    suffixes=("", "_sexo"),
)

df_covid_clean = df_covid_clean.merge(
    df_resultado[["clave", "descripcion", "id_resultado"]],
    how="left",
    left_on="RESULTADO",
    right_on="clave",
    suffixes=("", "_resultado"),
)

pd.DataFrame([
    {"etapa": "Integracion con dimensiones", "registros": len(df_covid_clean), "columnas": df_covid_clean.shape[1]},
])

,etapa,registros,columnas
0,Integracion con dimensiones,849638,58


## Validación de Resultados

In [16]:
resumen_final = pd.DataFrame([
    {"indicador": "Registros finales", "valor": f"{len(df_covid_clean):,}"},
    {"indicador": "Columnas finales", "valor": df_covid_clean.shape[1]},
    {"indicador": "Duplicados", "valor": int(df_covid_clean.duplicated().sum())},
    {"indicador": "id_caso unico", "valor": df_covid_clean["id_caso"].is_unique},
    {
        "indicador": "FECHA_DEF sin valor (despues de limpieza)",
        "valor": int(df_covid_clean["FECHA_DEF"].isna().sum()),
    },
    {
        "indicador": "Interpretacion FECHA_DEF vacia",
        "valor": "Casos sin defuncion registrada; valor esperado tras normalizar 9999-99-99",
    },
])

display(resumen_final)
df_covid_clean.head(10)

,indicador,valor
0,Registros finales,"849,638"
1,Columnas finales,58
2,Duplicados,0
3,id_caso unico,True
4,FECHA_DEF sin valor (despues de limpieza),795243
5,Interpretacion FECHA_DEF vacia,Casos sin defuncion registrada; valor esperado...


,SECTOR,ENTIDAD_UM,SEXO,ENTIDAD_RES,TIPO_PACIENTE,FECHA_INGRESO,FECHA_SINTOMAS,FECHA_DEF,INTUBADO,NEUMONIA,...,id_tipo_paciente,clave_entidad,entidad_federativa,id_entidad,clave_sexo,descripcion_sexo,id_sexo,clave_resultado,descripcion_resultado,id_resultado
0,4,9,2,9,1,2020-03-23,2020-03-22,NaT,97,2,...,1,9,CIUDAD DE MÉXICO,9,2,HOMBRE,2,1,Positivo SARS-CoV-2,1
1,4,9,2,9,2,2020-04-13,2020-04-04,NaT,2,1,...,2,9,CIUDAD DE MÉXICO,9,2,HOMBRE,2,1,Positivo SARS-CoV-2,1
2,4,8,1,8,2,2020-04-15,2020-04-10,2020-04-19,2,1,...,2,8,CHIHUAHUA,8,1,MUJER,1,1,Positivo SARS-CoV-2,1
3,4,30,1,30,1,2020-04-27,2020-04-17,NaT,97,2,...,1,30,VERACRUZ DE IGNACIO DE LA LLAVE,30,1,MUJER,1,1,Positivo SARS-CoV-2,1
4,3,15,2,15,2,2020-06-06,2020-06-01,NaT,2,1,...,2,15,MÉXICO,15,2,HOMBRE,2,1,Positivo SARS-CoV-2,1
5,3,15,1,15,1,2020-06-09,2020-06-05,NaT,97,2,...,1,15,MÉXICO,15,1,MUJER,1,1,Positivo SARS-CoV-2,1
6,3,15,2,15,1,2020-06-12,2020-06-08,2020-06-18,97,1,...,1,15,MÉXICO,15,2,HOMBRE,2,1,Positivo SARS-CoV-2,1
7,4,20,1,20,1,2020-03-31,2020-03-25,NaT,97,2,...,1,20,OAXACA,20,1,MUJER,1,1,Positivo SARS-CoV-2,1
8,4,14,2,14,2,2020-03-20,2020-03-20,2020-05-02,2,1,...,2,14,JALISCO,14,2,HOMBRE,2,1,Positivo SARS-CoV-2,1
9,4,31,2,31,1,2020-04-15,2020-04-13,NaT,97,2,...,1,31,YUCATÁN,31,2,HOMBRE,2,1,Positivo SARS-CoV-2,1


### Conclusión de la validación

El proceso de transformación dejó el dataset listo para etapas posteriores del modelo analítico. Se conservaron los registros de hechos, se generó un identificador único por caso, se estandarizaron códigos y fechas, y se enriqueció la información con catálogos dimensionales. Los valores vacíos en `FECHA_DEF` después de la limpieza corresponden al significado oficial del dataset y no representan pérdida incorrecta de información.


# Actividad Individual: Aplicación de la Práctica en el Entorno Profesional

## Cristian Gonzalez

# Reflexión sobre la aplicación de los conocimientos adquiridos

Durante esta semana reforcé la importancia de la transformación de datos dentro de un proceso ETL, ya que permite convertir información dispersa e inconsistente en datos confiables para el análisis. En mi trabajo como desarrollador de software en una empresa de telecomunicaciones, podría mejorar la consolidación de información proveniente de clientes, contratos, facturación, soporte técnico e infraestructura de red.

Los datos que manejo incluyen registros de clientes, estados de contratos, tickets de soporte, direcciones IP y datos de facturación. Entre los problemas más comunes se encuentran registros duplicados, campos incompletos, formatos de fecha diferentes, errores de digitación y códigos sin una descripción clara.

Para mejorar la calidad de estos datos se pueden aplicar funciones de limpieza y transformación, como eliminación de duplicados, tratamiento de valores nulos, normalización de textos, estandarización de fechas y validación de formatos. Estas acciones permiten obtener información más consistente y confiable.

Documentar cada paso en un notebook es importante porque facilita la trazabilidad, permite comprender las decisiones tomadas y favorece el trabajo colaborativo. Considero que esta práctica aporta directamente a la toma de decisiones, ya que reportes y análisis construidos sobre datos limpios generan resultados más precisos y útiles para la gestión del negocio.


## Rafael Espinosa

# Reflexión sobre la aplicación de los conocimientos adquiridos

Durante esta semana aprendí que la transformación de datos es una etapa fundamental del proceso ETL, ya que permite convertir datos provenientes de diferentes fuentes en información consistente, limpia y útil para el análisis. En mi entorno profesional podría aplicar estos conocimientos para mejorar la calidad de la información utilizada en proyectos de análisis de datos y generación de reportes, evitando errores ocasionados por registros incompletos, duplicados o con formatos inconsistentes.

Habitualmente manejo bases de datos provenientes de archivos CSV, sistemas transaccionales y consultas SQL. Entre los problemas más frecuentes se encuentran valores nulos, datos duplicados, diferencias en formatos de fechas, nombres y tipos de datos incorrectos. Mediante técnicas de limpieza, transformación y normalización, utilizando herramientas como Python y Pandas, es posible estandarizar la información y garantizar su confiabilidad.

Considero que documentar cada paso en un notebook es importante porque permite entender con facilidad qué se hizo durante el proceso y por qué se tomaron ciertas decisiones. Además, si en el futuro necesito revisar el trabajo o compartirlo con otra persona, será mucho más sencillo darle continuidad sin empezar desde cero. Llevar un registro ordenado de las transformaciones realizadas también ayuda a generar información más confiable, lo que facilita realizar análisis de mejor calidad y tomar decisiones con mayor seguridad.

## Oscar Amagua

# Reflexión sobre la aplicación de los conocimientos adquiridos

Dentro de mi formación en el área de ciberseguridad, un campo en el que se generan grandes volúmenes de datos provenientes de múltiples fuentes, como logs de sistemas, registros de accesos, tráfico de red, alertas de seguridad y eventos generados por herramientas de monitoreo, el análisis de los datos, se vuelve un proceso cotidiano y estos pueden presentarse en diferentes formatos, como bases de datos utilizadas en nuestra práctica( archivos JSON y CSV), por lo cual es necesario integrarlos eficientemente para un correcto análisis.

La correcta implementación de procesos ETL enfocados en ciberseguridad nos brinda múltiples características, como la centralización de datos, el monitoreo continuo de eventos y la detección temprana de anomalías. Además, que nos permite automatizar tareas repetitivas, reducir errores humanos, lo cual en su mayor medida permite mejorar la eficiencia en la gestión de incidentes de seguridad de manera que podamos enfocarnos en procesos de mayor criticidad.

Finalmente, gracias al análisis de estos datos, nos es posible tomar decisiones clave, como identificar patrones de comportamiento sospechoso, detectar accesos no autorizados, prevenir ataques cibernéticos y mejorar las políticas de seguridad. En conclusión, la aplicación de los procesos y actividades desarrolladas en clase, nos permite fortalecer nuestra comprensión y análisis en áreas o procesos que impactan directamente sobre la seguridad informática de nuestra organización, permitiendo responder de manera más rápida y efectiva ante posibles amenazas.